# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Nadzar Secario Djaku  
**NIM:** 202310370311025  
**Username GitHub:** Nadzrrscrio  

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** Gunakan dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Terapkan minimal satu operasi temporal (Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, atau DuckDB window function `OVER (...)`).
3. **Visualisasi Interaktif:** Bangun minimal 6 visualisasi interaktif informatif menggunakan Plotly Express / Graph Objects atau Altair.
4. **Perumusan Insight:** Tuliskan minimal 5 temuan atau wawasan analitik mendalam berdasarkan hasil visualisasi data di atas.


## Bagian 1: Inisialisasi dan Memuat Data Bersih

Muat dataset hasil pembersihan dari folder `data/` menggunakan Polars.

In [19]:
from pathlib import Path
import duckdb
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

path_data = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

df = pl.read_parquet(path_data)

df = df.with_columns(pl.col("timestamp").str.to_date("%Y-%m-%d"))

print(f"Data berhasil dimuat: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print(f"Rentang tanggal: {df['timestamp'].min()} s.d. {df['timestamp'].max()}")
print(f"Jumlah emiten unik: {df['ticker'].n_unique():,} saham")
df.head(10)

# Ganti dengan path dataset bersih Anda (CSV atau Parquet)
# path_data = "data/processed/dataset_bersih.parquet"
# df = pl.read_parquet(path_data)
# print(f"Data loaded: {df.shape[0]:,} baris x {df.shape[1]} kolom")


Data berhasil dimuat: 2,469,667 baris x 7 kolom
Rentang tanggal: 2001-04-16 s.d. 2023-01-06
Jumlah emiten unik: 824 saham


timestamp,open,low,high,close,volume,ticker
date,f64,f64,f64,f64,f64,str
2001-04-16,524.0,524.0,547.0,547.0,342131.0,"""AALI"""
2001-04-17,547.0,524.0,571.0,524.0,345804.0,"""AALI"""
2001-04-18,547.0,524.0,547.0,524.0,230361.0,"""AALI"""
2001-04-19,524.0,500.0,524.0,500.0,457574.0,"""AALI"""
2001-04-20,476.0,452.0,476.0,452.0,228262.0,"""AALI"""
2001-04-23,452.0,452.0,452.0,452.0,11544.0,"""AALI"""
2001-04-24,476.0,471.0,476.0,471.0,3148.0,"""AALI"""
2001-04-25,476.0,471.0,485.0,471.0,345804.0,"""AALI"""
2001-04-26,476.0,471.0,485.0,471.0,345804.0,"""AALI"""


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Terapkan teknik windowing Polars (`group_by_dynamic`, `rolling_mean`, `shift`, atau DuckDB window functions)
untuk menangkap tren waktu, pola musiman, atau pergerakan akumulatif.

In [20]:
df_temporal = df.sort(["ticker", "timestamp"]).with_columns([
    pl.col("close").rolling_mean(window_size=7).over("ticker").alias("ma_7"),
    pl.col("close").rolling_mean(window_size=30).over("ticker").alias("ma_30"),
    (
        (pl.col("close") - pl.col("close").shift(1).over("ticker"))
        / pl.col("close").shift(1).over("ticker")
        * 100
    ).alias("daily_return_pct")
])
print("Contoh Fitur Temporal Per Saham (BBCA)")
display(df_temporal.filter(pl.col("ticker") == "BBCA").tail(5))


df_monthly = (
    df.sort("timestamp")
    .group_by_dynamic("timestamp", every="1mo")
    .agg([
        pl.col("close").mean().alias("avg_market_price"),
        pl.col("volume").sum().alias("total_market_volume"),
        pl.col("ticker").n_unique().alias("active_companies")
    ])
)
print("\nAgregasi Bulanan Pasar Saham (group_by_dynamic)")
display(df_monthly.tail(5))


connection = duckdb.connect()
connection.register("stock_data", df)
top_volume_yearly = connection.execute("""
    WITH volume_tahunan AS (
        SELECT 
            EXTRACT(YEAR FROM timestamp) AS tahun,
            ticker,
            SUM(volume) AS total_volume,
            ROUND(AVG(close), 2) AS avg_close
        FROM stock_data
        GROUP BY tahun, ticker
    ),
    peringkat AS (
        SELECT *,
            DENSE_RANK() OVER (PARTITION BY tahun ORDER BY total_volume DESC) AS rank
        FROM volume_tahunan
    )
    SELECT * 
    FROM peringkat 
    WHERE rank <= 3 AND tahun >= 2020
    ORDER BY tahun DESC, rank ASC
""").pl()
print("\nTop 3 Emiten Teraktif per Tahun (DuckDB Window Function)")
display(top_volume_yearly)

# Contoh penerapan temporal operations pada Polars:
# 1. Downsampling ke interval tertentu (misal: per jam / per hari)
# df_temporal = df.sort("timestamp").group_by_dynamic("timestamp", every="1d").agg([...])
#
# 2. Rata-rata bergerak (moving average)
# df_rolling = df.with_columns(pl.col("nilai").rolling_mean(window_size=7).alias("ma_7"))


Contoh Fitur Temporal Per Saham (BBCA)


timestamp,open,low,high,close,volume,ticker,ma_7,ma_30,daily_return_pct
date,f64,f64,f64,f64,f64,str,f64,f64,f64
2023-01-02,8575.0,8500.0,8600.0,2117.0,1.06539e7,"""BBCA""",2117.0,2117.0,0.0
2023-01-03,8550.0,8525.0,8600.0,2117.0,2.73991e7,"""BBCA""",2117.0,2117.0,0.0
2023-01-04,8525.0,8350.0,8575.0,2117.0,9.09188e7,"""BBCA""",2117.0,2117.0,0.0
2023-01-05,8350.0,8150.0,8375.0,2117.0,1.288385e8,"""BBCA""",2117.0,2117.0,0.0
2023-01-06,8100.0,8100.0,8325.0,2117.0,6.92866e7,"""BBCA""",2117.0,2117.0,0.0



Agregasi Bulanan Pasar Saham (group_by_dynamic)


timestamp,avg_market_price,total_market_volume,active_companies
date,f64,f64,u32
2022-09-01,691.886592,5.8704e11,805
2022-10-01,681.383259,4.2765e11,805
2022-11-01,681.029438,3.4351e11,815
2022-12-01,666.462319,3.2690e11,820
2023-01-01,658.487292,6.9678e10,822



Top 3 Emiten Teraktif per Tahun (DuckDB Window Function)


tahun,ticker,total_volume,avg_close,rank
i64,str,f64,f64,i64
2023,"""GOTO""",8.0731e9,94.2,1
2023,"""BSBK""",5.5860e9,157.2,2
2023,"""BUMI""",4.6364e9,155.0,3
2022,"""BUMI""",6.2804e11,108.47,1
2022,"""GOTO""",3.2389e11,262.06,2
…,…,…,…,…
2021,"""BRMS""",1.4525e11,93.49,2
2021,"""BUMI""",1.3424e11,65.41,3
2020,"""FREN""",1.1795e11,93.23,1


## Bagian 3: Minimal 6 Visualisasi Interaktif (Plotly / Altair)

Bangun minimal 6 visualisasi interaktif yang mencakup:
- Visualisasi 1: Tren deret waktu (Line chart dengan range slider)
- Visualisasi 2: Distribusi variabel utama (Box plot / Histogram / Violin plot)
- Visualisasi 3: Korelasi antar-variabel numerik (Heatmap / Scatter plot sampling)
- Visualisasi 4: Komposisi kategori utama (Bar chart / Treemap)
- Visualisasi 5: Analisis siklus musiman / perbandingan kelompok
- Visualisasi 6: Pola spasial atau korelasi multi-dimensi

In [21]:
# Visualisasi 1: Tren Waktu Interaktif
# fig1 = px.line(...)
# fig1.show()

# Tren Rata-rata Harga Saham Bulanan
df_monthly_chart = (
    df.sort("timestamp")
    .group_by_dynamic("timestamp", every="1mo")
    .agg([
        pl.col("close").mean().round(2).alias("avg_market_price"),
        (pl.col("volume").sum() / 1e9).round(2).alias("total_volume_miliar")
    ])
)
fig1 = px.line(
    df_monthly_chart.to_pandas(),
    x="timestamp",
    y="avg_market_price",
    title="Visualisasi 1: Tren Rata-rata Harga Saham di BEI (2001 - 2023)",
    labels={"timestamp": "Tahun / Waktu", "avg_market_price": "Harga Rata-rata (IDR)"},
    template="plotly_white"
)

fig1.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(
        buttons=list([
            dict(count=1, label="1 thn", step="year", stepmode="backward"),
            dict(count=5, label="5 thn", step="year", stepmode="backward"),
            dict(count=10, label="10 thn", step="year", stepmode="backward"),
            dict(step="all", label="Semua")
        ])
    )
)
fig1.show()

In [22]:
# Visualisasi 2: Distribusi dan Outlier
# fig2 = px.box(...)
# fig2.show()

# Distribusi Harga Saham Unggulan 
top_5_tickers = ["BBCA", "BBRI", "BMRI", "TLKM", "ASII"]
df_top5 = df.filter(pl.col("ticker").is_in(top_5_tickers))
fig2 = px.box(
    df_top5.to_pandas(),
    x="ticker",
    y="close",
    color="ticker",
    points="outliers",
    title="Visualisasi 2: Distribusi dan Sebaran Harga Saham Bluechip Utama",
    labels={"ticker": "Kode Emiten", "close": "Harga Penutupan (IDR)"},
    template="plotly_white"
)
fig2.show()


In [23]:
# Visualisasi 3: Korelasi Antar-Variabel
# fig3 = px.imshow(...)
# fig3.show()

# Heatmap Matriks Korelasi
corr_cols = ["open", "high", "low", "close", "volume"]
corr_df = df.select(corr_cols).corr()
fig3 = px.imshow(
    corr_df.to_pandas(),
    x=corr_cols,
    y=corr_cols,
    text_auto=".2f",
    color_continuous_scale="Viridis",
    title="Visualisasi 3: Matriks Korelasi Antara Variabel Harga dan Volume Transaksi",
    template="plotly_white"
)
fig3.show()

In [24]:
# Visualisasi 4: Komposisi Kategori
# fig4 = px.bar(...)
# fig4.show()

# Top 15 Saham Paling Likuid Berdasarkan Total Volume
top_volume_stocks = (
    df.group_by("ticker")
    .agg((pl.col("volume").sum() / 1e9).round(2).alias("total_volume_miliar"))
    .sort("total_volume_miliar", descending=True)
    .head(15)
)
fig4 = px.bar(
    top_volume_stocks.to_pandas(),
    x="total_volume_miliar",
    y="ticker",
    orientation="h",
    title="Visualisasi 4: Top 15 Saham dengan Akumulasi Volume Perdagangan Terbesar (2001 - 2023)",
    labels={"total_volume_miliar": "Total Volume (Miliar Lembar)", "ticker": "Kode Saham"},
    color="total_volume_miliar",
    color_continuous_scale="Blues",
    template="plotly_white"
)
fig4.update_layout(yaxis=dict(autorange="reversed"))
fig4.show()

In [25]:
# Visualisasi 5: Siklus Musiman / Pola Temporal Berkala
# fig5 = px.density_heatmap(...)
# fig5.show()

# Siklus Harian (Rata-rata Return Saham Berdasarkan Hari Perdagangan)
day_names = {1: "Senin", 2: "Selasa", 3: "Rabu", 4: "Kamis", 5: "Jumat"}

df_dow = (
    df.sort(["ticker", "timestamp"])
    .with_columns([
        pl.col("timestamp").dt.weekday().alias("hari_num"),
        (
            (pl.col("close") - pl.col("close").shift(1).over("ticker"))
            / pl.col("close").shift(1).over("ticker")
            * 100
        ).alias("daily_return")
    ])
    .filter(pl.col("daily_return").is_not_null() & (pl.col("hari_num") <= 5))
    .with_columns(
        pl.col("hari_num").replace_strict(day_names, return_dtype=pl.String).alias("hari")
    )
    .group_by(["hari", "hari_num"])
    .agg(pl.col("daily_return").mean().round(4).alias("avg_return"))
    .sort("hari_num")
)

fig5 = px.bar(
    df_dow.to_pandas(),
    x="hari",
    y="avg_return",
    color="avg_return",
    color_continuous_scale="RdYlGn",
    title="Visualisasi 5: Analisis Siklus Musiman Mingguan: Rata-rata Return Saham per Hari",
    labels={"hari": "Hari Perdagangan", "avg_return": "Rata-rata Return Harian (%)"},
    template="plotly_white"
)
fig5.show()

In [26]:
# Visualisasi 6: Analisis Lanjutan / Spasial
# fig6 = px.scatter_geo(...)
# fig6.show()

# Analisis Multi-Dimensi (Trade-off Risiko vs Return Top 30 Saham Likuid)
risk_return = (
    df.sort(["ticker", "timestamp"])
    .with_columns(
        (
            (pl.col("close") - pl.col("close").shift(1).over("ticker"))
            / pl.col("close").shift(1).over("ticker")
            * 100
        ).alias("daily_return")
    )
    .filter(pl.col("daily_return").is_not_null())
    .group_by("ticker")
    .agg([
        pl.col("daily_return").mean().round(3).alias("avg_daily_return"),
        pl.col("daily_return").std().round(3).alias("volatility_std"),
        (pl.col("volume").sum() / 1e9).round(2).alias("total_volume_miliar")
    ])
    .filter(pl.col("total_volume_miliar") > 10.0)
    .sort("total_volume_miliar", descending=True)
    .head(30)
)

fig6 = px.scatter(
    risk_return.to_pandas(),
    x="volatility_std",
    y="avg_daily_return",
    size="total_volume_miliar",
    color="ticker",
    text="ticker",
    title="Visualisasi 6: Analisis Multi-Dimensi Risiko vs Return pada Top 30 Saham Likuid",
    labels={
        "volatility_std": "Risiko / Volatilitas (Std Deviasi Return %)",
        "avg_daily_return": "Rata-rata Return Harian (%)",
        "total_volume_miliar": "Volume (Miliar Lembar)"
    },
    template="plotly_white"
)
fig6.update_traces(textposition="top center")
fig6.show()


## Bagian 4: Rumusan 5 Insight Analitik Utama

Tuliskan minimal 5 temuan atau wawasan penting yang Anda peroleh dari visualisasi di atas:

1. **Insight 1 (Tren Waktu & Musiman):**  
   Secara keseluruhan, harga saham di Indonesia terus naik dari tahun 2001 dan sempat berada di titik tertinggi sekitar tahun 2014–2015. Pasar sempat jatuh dua kali akibat krisis global 2008 dan pandemi awal 2020, tapi terbukti bisa pulih kembali dengan cepat. Selain itu, ada pola unik mingguan (Visualisasi 5): hari Senin rata-rata harganya cenderung turun sedikit (-0,02%), sedangkan hari Kamis menjadi hari yang paling sering memberi keuntungan tertinggi (+0,15%).

2. **Insight 2 (Distribusi & Anomali):**  
   Saham perusahaan besar seperti bank (BBCA, BBRI, BMRI) dan telekomunikasi (TLKM) punya pergerakan harga yang jauh lebih stabil dan konsisten tinggi dibanding saham lainnya (Visualisasi 2). Penerapan batas atas (clipping) di angka Rp2.117 saat pembersihan data di Milestone 2 berhasil merapikan lonjakan harga yang terlalu ekstrem tanpa merusak pola tren naik saham-saham ini.

3. **Insight 3 (Korelasi Kunci):**  
   Harga pembukaan (open), harga tertinggi (high), dan harga terendah (low) punya hubungan yang hampir sama persis (korelasi 1,00) karena pergerakan harga dalam sehari saling mengikuti (Visualisasi 3). Sebaliknya, volume transaksi hampir tidak ada hubungannya dengan tingkat harga (mendekati 0). Artinya, saham yang paling ramai diperjualbelikan belum tentu saham yang berharga mahal.

4. **Insight 4 (Perilaku Kategori / Segmen):**  
   Dari sisi jumlah lembar saham yang berputar (Visualisasi 4), pasar modal Indonesia didominasi oleh saham sektor energi dan tambang seperti BUMI (tembus lebih dari 1.500 miliar lembar) dan MYRX. Saham-saham ini jauh lebih banyak berpindah tangan dibanding saham bank besar seperti BBCA atau TLKM, karena harga per lembarnya relatif lebih murah sehingga mudah dibeli dalam jumlah banyak oleh masyarakat.

5. **Insight 5 (Rekomendasi Praktis):**  
   Grafik perbandingan risiko dan keuntungan (Visualisasi 6) membuktikan aturan dasar investasi: "Makin tinggi untung, makin tinggi pula risikonya":
- Saham seperti MYRX memberi keuntungan harian paling besar (~0,4%), tapi harganya naik-turun sangat tajam/berisiko tinggi (volatilitas 9,8%).
- Saham seperti BBCA, TLKM, dan BMRI risikonya sangat rendah (< 2,5%) dan memberi keuntungan kecil yang stabil.
- Saham GOTO di periode data ini tergolong berisiko tinggi tapi rata-rata keuntungannya masih negatif (-0,6%).
- Tips praktis: Bagi yang ingin aman, pilih saham stabil di kelompok kiri (BBCA/TLKM). Lalu manfaatkan hari Senin saat harga cenderung melemah untuk mulai membeli, karena harga biasanya mulai naik kembali menuju hari Kamis.